In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviewsPA5") \
    .getOrCreate()

print("Spark session created successfully")
print(spark)

Spark session created successfully


In [3]:
print("PySpark version check:")
print(spark.version)

PySpark version check:
4.0.2


In [4]:
from google.colab import files
files.upload()

Saving amz_ca_total_products_data_processed.csv to amz_ca_total_products_data_processed.csv


In [32]:
from pyspark.sql.functions import when, col

df = spark.read.csv(
    "amz_ca_total_products_data_processed.csv",
    header=True,
    inferSchema=False,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True
)

df.select("price").show(20, truncate=False)

+-----+
|price|
+-----+
|47.69|
|10.99|
|25.99|
|21.99|
|18.99|
|15.99|
|27.99|
|9.99 |
|15.95|
|29.99|
|12.99|
|32.99|
|23.99|
|19.97|
|9.71 |
|53.97|
|81.99|
|66.86|
|21.99|
|20.99|
+-----+
only showing top 20 rows


In [7]:
print("Number of columns:", len(df.columns))
print(df.columns)

Number of columns: 1
['"asin","title","imgUrl","productURL","stars","reviews","price","listPrice","categoryName","isBestSeller","boughtInLastMonth"']


In [8]:
df.printSchema()

root
 |-- "asin","title","imgUrl","productURL","stars","reviews","price","listPrice","categoryName","isBestSeller","boughtInLastMonth": string (nullable = true)



In [9]:
print("Row count:", df.count())


Row count: 2165926


In [11]:
raw = spark.read.text("amz_ca_total_products_data_processed.csv")
raw.show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                                                                                                                   |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
with open("amz_ca_total_products_data_processed.csv", "r") as f:
    text = f.read()

text = text.replace("\\t", "\t")

with open("amz_fixed.tsv", "w") as f:
    f.write(text)

print("Fixed file created")

Fixed file created


In [14]:
df = spark.read.csv(
    "amz_fixed.tsv",
    header=True,
    sep="\t",
    inferSchema=True
)

df.show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|"asin","title","imgUrl","productURL","stars","reviews","price","listPrice","categoryName","isBestSeller","boughtInLastMonth"                                                                                                                                                                                                                            |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [15]:
print("Number of columns:", len(df.columns))
print(df.columns)
df.printSchema()
print("Row count:", df.count())

Number of columns: 1
['"asin","title","imgUrl","productURL","stars","reviews","price","listPrice","categoryName","isBestSeller","boughtInLastMonth"']
root
 |-- "asin","title","imgUrl","productURL","stars","reviews","price","listPrice","categoryName","isBestSeller","boughtInLastMonth": string (nullable = true)

Row count: 2165926


In [35]:
from pyspark.sql.functions import when, col

df.printSchema()

df = df.withColumn(
    "price_bucket",
    when(col("listPrice") < 10, "LOW")
    .when(col("listPrice") < 30, "MID")
    .when(col("listPrice") < 100, "HIGH")
    .otherwise("PREMIUM")
)

df = df.withColumn(
    "rating_category",
    when(col("stars") >= 4, "GOOD")
    .when(col("stars") >= 2, "AVERAGE")
    .otherwise("LOW_OR_NONE")
)

df.select("stars", "listPrice", "price_bucket", "rating_category").show(10, truncate=False)

root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- imgUrl: string (nullable = true)
 |-- productURL: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- reviews: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- listPrice: double (nullable = true)
 |-- categoryName: string (nullable = true)
 |-- isBestSeller: string (nullable = true)
 |-- boughtInLastMonth: integer (nullable = true)
 |-- price_bucket: string (nullable = false)
 |-- rating_category: string (nullable = false)

+-----+---------+------------+---------------+
|stars|listPrice|price_bucket|rating_category|
+-----+---------+------------+---------------+
|4.4  |0.0      |LOW         |GOOD           |
|3.8  |0.0      |LOW         |AVERAGE        |
|4.0  |27.99    |MID         |GOOD           |
|4.5  |30.99    |HIGH        |GOOD           |
|4.2  |0.0      |LOW         |GOOD           |
|4.5  |0.0      |LOW         |GOOD           |
|4.3  |0.0      |LOW         |GO

In [ ]:
price_bucket_summary = df.groupBy("price_bucket").count().orderBy("count", ascending=False)
price_bucket_summary.show()

In [ ]:
top_categories = df.groupBy("categoryName").count().orderBy("count", ascending=False)
top_categories.show(10, truncate=False)

In [ ]:
import matplotlib.pyplot as plt

top_categories_pd = top_categories.limit(10).toPandas()

plt.figure(figsize=(10,6))
plt.barh(top_categories_pd["categoryName"], top_categories_pd["count"])
plt.xlabel("Count")
plt.ylabel("Category")
plt.title("Top 10 Product Categories")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
rating_summary.coalesce(1).write.mode("overwrite").option("header", True).csv("rating_summary_pyspark")
price_bucket_summary.coalesce(1).write.mode("overwrite").option("header", True).csv("price_bucket_summary_pyspark")
top_categories.limit(10).coalesce(1).write.mode("overwrite").option("header", True).csv("top_categories_pyspark")

In [ ]:
rating_summary.show()
price_bucket_summary.show()
top_categories.show(10, truncate=False)

Sprint 5 PySpark Workflow
- Set up PySpark in Google Colab
- Loaded and verified normalized dataset
- Created derived columns: price_bucket and rating_category
- Built PySpark summary tables using groupBy
- Generated a figure for category-level insight
- Saved outputs for reproducibility

In [17]:
print("Spark version:", spark.version)
print("Total rows:", df.count())
print("Total columns:", len(df.columns))

Spark version: 4.0.2
Total rows: 2165926
Total columns: 1


In [34]:
df.groupBy("rating_category").count().show()

+---------------+-------+
|rating_category|  count|
+---------------+-------+
|        AVERAGE| 195805|
|           GOOD|1110346|
|    LOW_OR_NONE| 859775|
+---------------+-------+



In [33]:
from pyspark.sql.functions import when, col, regexp_replace
from pyspark.sql.types import DoubleType, IntegerType


df = df.withColumn("price", regexp_replace(col("price"), "[^0-9.]", "").cast(DoubleType())) \
       .withColumn("listPrice", regexp_replace(col("listPrice"), "[^0-9.]", "").cast(DoubleType())) \
       .withColumn("stars", regexp_replace(col("stars"), "[^0-9.]", "").cast(DoubleType())) \
       .withColumn("reviews", regexp_replace(col("reviews"), "[^0-9]", "").cast(IntegerType())) \
       .withColumn("boughtInLastMonth", regexp_replace(col("boughtInLastMonth"), "[^0-9]", "").cast(IntegerType()))

df = df.withColumn(
    "price_bucket",
    when(col("price") < 10, "LOW")
    .when(col("price") < 30, "MID")
    .when(col("price") < 100, "HIGH")
    .otherwise("PREMIUM")
)


df = df.withColumn(
    "rating_category",
    when(col("stars") >= 4, "GOOD")
    .when(col("stars") >= 2, "AVERAGE")
    .otherwise("LOW_OR_NONE")
)

df.select("stars", "price", "price_bucket", "rating_category").show(10, truncate=False)

+-----+-----+------------+---------------+
|stars|price|price_bucket|rating_category|
+-----+-----+------------+---------------+
|4.4  |47.69|HIGH        |GOOD           |
|3.8  |10.99|MID         |AVERAGE        |
|4.0  |25.99|MID         |GOOD           |
|4.5  |21.99|MID         |GOOD           |
|4.2  |18.99|MID         |GOOD           |
|4.5  |15.99|MID         |GOOD           |
|4.3  |27.99|MID         |GOOD           |
|4.0  |9.99 |LOW         |GOOD           |
|4.5  |15.95|MID         |GOOD           |
|4.4  |29.99|MID         |GOOD           |
+-----+-----+------------+---------------+
only showing top 10 rows


In [24]:
df.printSchema()

root
 |-- asin: string (nullable = true)
 |-- title: string (nullable = true)
 |-- imgUrl: string (nullable = true)
 |-- productURL: string (nullable = true)
 |-- stars: string (nullable = true)
 |-- reviews: string (nullable = true)
 |-- price: string (nullable = true)
 |-- listPrice: string (nullable = true)
 |-- categoryName: string (nullable = true)
 |-- isBestSeller: string (nullable = true)
 |-- boughtInLastMonth: string (nullable = true)

